# BrainRoute Validation Figures: Blue Theme and 3D Molecular Property Space

This notebook regenerates reviewer-facing model-performance bar plots using a blue theme with capped standard-deviation error bars. It also creates a 3D scatter plot of all molecules in the PaDEL + Morgan + ChemBERTa feature view using PaDEL TPSA, XLogP, and molecular weight.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name != "brainroute_ml_validation":
    ROOT = ROOT / "brainroute_ml_validation"

REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
PROCESSED = ROOT / "data" / "processed"
FIGURES.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk")
BLUE_PALETTE = sns.color_palette("Blues", n_colors=8)[2:]

perf = pd.read_csv(REPORTS / "model_performance_all_splits.csv")
perf.head()

## Blue Bar Plots With Capped Standard-Deviation Error Bars

The error bars below use the standard deviation across folds/splits. The `capsize` option creates the small horizontal lines at the top and bottom of each standard-deviation bar.

In [ ]:
def blue_barplot(data, metric, title, output_name):
    data = data.copy()
    order = ["logistic_regression", "knn", "random_forest", "extra_trees", "lightgbm", "xgboost"]
    hue_order = ["padel", "morgan", "padel_morgan", "embeddings", "padel_morgan_embeddings"]
    hue_order = [h for h in hue_order if h in set(data["feature_view"])]
    palette = sns.color_palette("Blues", n_colors=len(hue_order) + 3)[2 : 2 + len(hue_order)]

    fig, ax = plt.subplots(figsize=(13, 6.5))
    sns.barplot(
        data=data,
        x="model",
        y=metric,
        hue="feature_view",
        order=[m for m in order if m in set(data["model"])],
        hue_order=hue_order,
        errorbar="sd",
        capsize=0.14,
        err_kws={"linewidth": 1.4, "color": "#111827"},
        palette=palette,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(metric.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=30)
    for tick in ax.get_xticklabels():
        tick.set_horizontalalignment("right")
    ax.legend(title="Feature view", fontsize=9, title_fontsize=10, frameon=False, loc="best")
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(FIGURES / output_name, dpi=240, bbox_inches="tight")
    plt.show()

scaffold = perf[perf["split"].str.startswith("scaffold_cv_fold", na=False)]
duplicate = perf[perf["split"].str.startswith("duplicate_aware_seed", na=False)]

for metric in ["balanced_accuracy", "auprc", "roc_auc", "mcc", "f1"]:
    blue_barplot(
        scaffold,
        metric,
        f"Scaffold-CV {metric.replace('_', ' ').title()} By Model And Feature View",
        f"blue_scaffold_cv_{metric}_by_model_feature_view.png",
    )

for metric in ["balanced_accuracy", "auprc"]:
    blue_barplot(
        duplicate,
        metric,
        f"Duplicate-Aware Repeated Split {metric.replace('_', ' ').title()}",
        f"blue_duplicate_aware_{metric}_by_model_feature_view.png",
    )

## 3D Molecular Property Space

Each dot is one molecule from the PaDEL + Morgan + ChemBERTa feature matrix. The axes are PaDEL TPSA, XLogP, and molecular weight. Red indicates BBB-, and blue indicates BBB+.

In [ ]:
feature_path = PROCESSED / "features_padel_morgan_embeddings.csv"
index_path = PROCESSED / "features_padel_morgan_embeddings_index.csv"

property_columns = ["padel__TPSA", "padel__XLogP", "padel__MW"]
X_props = pd.read_csv(feature_path, usecols=property_columns)
idx = pd.read_csv(index_path, usecols=["molecule_id", "label"])
plot_df = pd.concat([idx, X_props], axis=1)
plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["label"])
missing_counts = plot_df[property_columns].isna().sum()
plot_df[property_columns] = plot_df[property_columns].fillna(plot_df[property_columns].median())

colors = plot_df["label"].map({0: "#d62728", 1: "#1f77b4"})

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    plot_df["padel__TPSA"],
    plot_df["padel__XLogP"],
    plot_df["padel__MW"],
    c=colors,
    s=8,
    alpha=0.55,
    linewidths=0,
)
ax.set_xlabel("PaDEL TPSA")
ax.set_ylabel("PaDEL XLogP")
ax.set_zlabel("PaDEL Molecular Weight")
ax.set_title("BBB Molecules In PaDEL Property Space")
ax.view_init(elev=24, azim=38)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], marker="o", color="w", label="BBB-", markerfacecolor="#d62728", markersize=8),
    Line2D([0], [0], marker="o", color="w", label="BBB+", markerfacecolor="#1f77b4", markersize=8),
]
ax.legend(handles=legend_handles, loc="upper right", frameon=False)

fig.tight_layout()
fig.savefig(FIGURES / "molecule_property_space_3d_tpsa_xlogp_mw.png", dpi=260, bbox_inches="tight")
plt.show()

print(f"Plotted {len(plot_df):,} molecules")
print("Median-imputed plotting-axis values:")
print(missing_counts)